In [24]:
import os
"""
EF-02 | Relevance + Category + Sentiment Classifier
----------------------------------------------------
Paste these cells into your existing notebook.
Designed to match the batching pattern from your working
multi_model_sentiment_batch() function.
"""

# ════════════════════════════════════════════════════════════
# CELL 1 — Imports & Config
# ════════════════════════════════════════════════════════════

import pandas as pd
import json
import time
import re
from openai import OpenAI

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
BATCH_SIZE   = 25           # 25 rows per API call — proven safe in your system
MODEL        = "llama-3.1-8b-instant"
SLEEP_SEC    = 1.5          # seconds between batches

groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY
)

SYSTEM_PROMPT = """You are a financial news classifier for Tanzania.
You return ONLY a valid JSON array. No preamble, no explanation, no markdown fences.
Every object in the array must be complete and the array must be properly closed with ]."""


# ════════════════════════════════════════════════════════════
# CELL 2 — Prompt Builder
# ════════════════════════════════════════════════════════════

def build_prompt(batch):
    """
    batch: list of dicts [{"id": 0, "headline": "..."}, ...]
    Numbers headlines 1-based within the batch only.
    """
    lines = "\n".join(
        f"{i+1}. {item['headline']}"
        for i, item in enumerate(batch)
    )

    return f"""Classify each Tanzanian financial news headline below.

Return a JSON array with one object per headline, in the same order.
Each object must have exactly these fields:

- "pos": integer — the headline number (1, 2, 3 ...)
- "relevant": true or false — true if the headline relates to Tanzania's economy,
  finance, business, trade, banking, investment, currency, inflation, energy costs,
  or agricultural markets. false for pure politics, sports, crime, entertainment,
  health, or social affairs with no economic angle.
- "category": one of these strings if relevant is true, otherwise null —
    "Forex"        exchange rates, shilling/dollar, currency reserves
    "Policy"       BOT, IMF, World Bank, govt economic policy, interest rates, inflation stats
    "Banking"      commercial banks (CRDB, NBC), mobile money, insurance, fintech
    "Trade"        imports/exports, AfCFTA, ports, counterfeit goods, tariffs
    "Agriculture"  farming, crops, tea, sugar, dairy, food prices
    "Energy"       TANESCO, power supply, electricity costs, fuel prices
    "Transport"    SGR, railways, roads, ports in logistics context
    "Investment"   FDI, PPP, new projects, business expansion
    "General"      financially relevant but fits none of the above
- "sentiment": one of "Positive", "Negative", "Neutral" if relevant is true, otherwise null.
  Classify by IMPLIED ECONOMIC OUTCOME, not emotional tone:
    Positive = growth, stability, record highs, new investment, rate holds amid stability
    Negative = decline, weakening, shortage, rate hikes amid crisis, job losses
    Neutral  = announcements, reports, projections with no clear positive/negative outcome
- "reason": 4-6 words explaining your decision (used for debugging, will be removed later)

Return ONLY the JSON array. No other text. Close the array with ].

Headlines:
{lines}"""


# ════════════════════════════════════════════════════════════
# CELL 3 — Single Batch Classifier
# ════════════════════════════════════════════════════════════

def classify_batch(batch, batch_num, total_batches, max_retries=3):
    """
    Sends one batch of up to 25 headlines.
    Returns list of result dicts with original row IDs restored.
    Mirrors retry logic from your multi_model_sentiment_batch().
    """
    prompt = build_prompt(batch)

    for attempt in range(1, max_retries + 1):
        try:
            response = groq_client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0,
                max_tokens=2500   # 25 rows × ~80 tokens each = ~2000, headroom added
            )

            raw = response.choices[0].message.content.strip()

            # Strip markdown fences if model adds them
            clean = re.sub(r'```(?:json)?|```', '', raw).strip()

            parsed = json.loads(clean)

            # Validate — must be a list with same length as batch
            if not isinstance(parsed, list):
                raise ValueError(f"Expected list, got {type(parsed)}")
            if len(parsed) != len(batch):
                raise ValueError(f"Got {len(parsed)} results for {len(batch)} headlines")

            # Remap pos (1-based within batch) back to original row id
            for i, result in enumerate(parsed):
                result['id'] = batch[i]['id']   # restore the global row id
                result.pop('pos', None)          # remove the local position field

            print(f"  Batch {batch_num}/{total_batches} — OK ({len(parsed)} results)")
            return parsed

        except json.JSONDecodeError:
            tail = raw[-80:] if len(raw) > 80 else raw
            print(f"  Batch {batch_num} attempt {attempt}: JSON cut off. Ends: ...{repr(tail)}")
            time.sleep(2)

        except ValueError as e:
            print(f"  Batch {batch_num} attempt {attempt}: Validation error — {e}")
            time.sleep(2)

        except Exception as e:
            print(f"  Batch {batch_num} attempt {attempt}: API error — {e}")
            time.sleep(3)

    # All retries exhausted — return placeholder rows so merge doesn't break
    print(f"  Batch {batch_num}: FAILED after {max_retries} attempts. Filling with None.")
    return [
        {"id": item['id'], "relevant": None, "category": None,
         "sentiment": None, "reason": "batch_failed"}
        for item in batch
    ]


# ════════════════════════════════════════════════════════════
# CELL 4 — Main Runner
# ════════════════════════════════════════════════════════════

def run_classifier(input_csv, output_csv, batch_size=25, test_size=None):
    # 1. Load
    df = pd.read_csv(input_csv, engine='python', on_bad_lines='warn')
    print(f"Loaded: {len(df)} rows from {input_csv}")
    
    if not test_size:
        test_size=len(df)

    # 2. Sample if testing
    if test_size and test_size < len(df):
        df = df.sample(n=test_size, random_state=42).reset_index(drop=True)
        print(f"Sampled: {len(df)} rows for testing")
    else:
        df = df.reset_index(drop=True)
        print(f"Running on full dataset: {len(df)} rows")

    df['id'] = df.index  # global row id used to reassemble results

    # 3. Split into batches — exactly like your working code
    rows      = df[['id', 'headline']].to_dict('records')
    batches   = [rows[i:i+batch_size] for i in range(0, len(rows), batch_size)]
    total     = len(batches)
    print(f"\n{total} batches of up to {batch_size} headlines each")
    print(f"Estimated time: ~{total * SLEEP_SEC / 60:.1f} minutes\n")

    # 4. Loop through batches
    all_results = []
    for batch_num, batch in enumerate(batches, start=1):
        results = classify_batch(batch, batch_num, total)
        all_results.extend(results)

        if batch_num < total:
            time.sleep(SLEEP_SEC)   # polite delay between batches

    # 5. Merge results back onto original dataframe
    df_results = pd.DataFrame(all_results)
    df_out     = df.merge(df_results, on='id', how='left').drop(columns=['id'])

    # 6. Summary
    total_rows = len(df_out)
    n_rel      = df_out['relevant'].sum()
    n_irr      = total_rows - n_rel
    n_failed   = (df_out['reason'] == 'batch_failed').sum()

    print(f"\n{'='*50}")
    print(f"DONE")
    print(f"{'='*50}")
    print(f"  Total rows     : {total_rows}")
    print(f"  Relevant       : {int(n_rel)} ({n_rel/total_rows*100:.1f}%)")
    print(f"  Irrelevant     : {int(n_irr)} ({n_irr/total_rows*100:.1f}%)")
    if n_failed:
        print(f"  Failed batches : {n_failed} rows affected — rerun those manually")

    print(f"\n  Category breakdown (relevant only):")
    cats = df_out[df_out['relevant'] == True]['category'].value_counts()
    for cat, count in cats.items():
        print(f"    {cat:<14} {count}")

    print(f"\n  Sentiment breakdown (relevant only):")
    sents = df_out[df_out['relevant'] == True]['sentiment'].value_counts()
    for sent, count in sents.items():
        print(f"    {sent:<14} {count}")

    # 7. Save — keep reason for now, drop it before production
    df_out.to_csv(output_csv, index=False)
    print(f"\nSaved to {output_csv}")

    return df_out




In [47]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Run It
# ════════════════════════════════════════════════════════════
INPUT_CSV    = "tz_headlines_clean.csv"
OUTPUT_CSV   = "tz_headlines_labelled.csv"

df_labelled = run_classifier(
    input_csv  = INPUT_CSV,
    output_csv = OUTPUT_CSV,
    test_size  = False, # implying print everything 
    batch_size = BATCH_SIZE
)

# Quick peek
df_labelled[['date','headline','relevant','category','sentiment','reason']].head(20)

Loaded: 5684 rows from tz_headlines_clean.csv
Running on full dataset: 5684 rows

228 batches of up to 25 headlines each
Estimated time: ~5.7 minutes

  Batch 1/228 — OK (25 results)
  Batch 2/228 — OK (25 results)
  Batch 3/228 — OK (25 results)
  Batch 4/228 — OK (25 results)
  Batch 5/228 — OK (25 results)
  Batch 6/228 — OK (25 results)
  Batch 7/228 — OK (25 results)
  Batch 8/228 — OK (25 results)
  Batch 9/228 — OK (25 results)
  Batch 10/228 — OK (25 results)
  Batch 11/228 — OK (25 results)
  Batch 12/228 — OK (25 results)
  Batch 13/228 — OK (25 results)
  Batch 14/228 — OK (25 results)
  Batch 15/228 — OK (25 results)
  Batch 16/228 — OK (25 results)
  Batch 17/228 — OK (25 results)
  Batch 18/228 — OK (25 results)
  Batch 19/228 — OK (25 results)
  Batch 20/228 — OK (25 results)
  Batch 21/228 — OK (25 results)
  Batch 22/228 — OK (25 results)
  Batch 23/228 — OK (25 results)
  Batch 24/228 — OK (25 results)
  Batch 25/228 — OK (25 results)
  Batch 26/228 — OK (25 results)


,date,headline,relevant,category,sentiment,reason
0,2024-01-01,Economists’ advice to government as Tanzania welcomes 2024,False,None,None,pure politics
1,2024-01-01,Tanzania moves to bring down soaring sugar prices,True,Agriculture,Neutral,sugar prices announcement
2,2023-12-31,Disagreement on days set to climb Mt. Kilimanjaro,False,None,None,pure sports
3,2023-12-31,Why government’s directive on boarding schools flopped,False,None,None,pure politics
4,2023-12-30,TRC receives new locomotives and passenger cars,True,Transport,Neutral,new locomotives and cars
5,2023-12-30,Tanzania forecasts Sh22.5 trillion PPP boom by 2025,True,Investment,Positive,forecast of PPP boom
6,2023-12-29,Tanzania's trophy hunting industry under scrutiny,False,None,None,pure politics
7,2023-12-28,Samia receives third doctorate in 12 months,False,None,None,pure politics
8,2023-12-28,Tanzanians count down the days before SGR services commence,True,Transport,Neutral,SGR services commence
9,2023-12-27,Why cement prices are high despite surplus production,True,General,Negative,high cement prices despite surplus


In [49]:
pd.set_option("display.max_colwidth", None)  # show full text in all columns

df_labelled["relevant"] = df_labelled["relevant"].astype(bool)
bad = df_labelled[~df_labelled["relevant"]]


print(len(bad))
bad[["headline", "reason"]].sample(50)



1819


,headline,reason
1471,Netflix loses subscribers for first time in more than a decade,"pure business, no economic angle"
4547,‘Obey President’s directives on petty traders’,politics news
3560,Mwinyi calls for concerted efforts to combat illegal fishing,combating illegal fishing
386,TRA return confiscated goods to Kariakoo traders,"confiscated goods returned to traders, no economic angle"
4795,SBL wages full war on teens alcohol drinking,SBL campaign unrelated to economy
4193,Absa launches empowering Africa tomorrow,purely social affairs
5350,PM Majaliwa off to Korea for three-day visit,pure politics headline
5555,Bunge committee satisfied with TAEC office construction,Bunge committee satisfied
663,US investor who fell in love with Tanzania,no economic angle
1461,The untold story of the making of Tanganyika -Zanzibar union,"pure history, no economic angle"


In [44]:
# Isolate rows with Positive sentiment
positive = df_labelled[df_labelled["relevant"] == True]
print(len(positive))
# Show only headline and reason columns
positive[["headline","sentiment", "source","reason"]].sample(50)


683


,headline,sentiment,source,reason
373,Mkombozi Bank records Sh3billion profit after tax in 2021,Positive,The Citizen,Mkombozi Bank records profit
348,Kilimanjaro embarks on strategies to woo more investors,Positive,Daily News,Kilimanjaro investor attraction
551,AD: Access Bank Plc enters into acquisition agreements with Five Standard Chartered Shareholdings in Africa,Neutral,The Citizen,bank acquisition agreement
953,Tanzania receives first-ever cargo plane,Neutral,Daily News,cargo plane delivery
908,5G launch offers new hope for Tanzania’s tech strategy,Positive,The Citizen,5G launch offers new hope for Tanzania's tech strategy
452,EABC improves knowledge of women cross-border traders and youths,Neutral,Daily News,biodiversity as economy
849,Tanzania and Kenya accuse Uganda of stalling $60 billion lake project,Neutral,The Citizen,Tanzania and Kenya accuse Uganda of stalling $60 billion lake project
865,EPZA urges councils to allocate land for industries,Neutral,Daily News,EPZA land allocation
498,TIC records rise in domestic investments,Neutral,Daily News,investment interest
442,TIC to launch one stop electronic system centre,Neutral,Daily News,TIC to launch electronic system


In [43]:
df_labelled

,date,headline,url,source,scraped_on,year_month,relevant,category,sentiment,reason,region
0,2024-09-24,Retire comfortably: The power of passive income,https://dailynews.co.tz/retire-comfortably-the-power-of-passive-income/,Daily News,2026-05-20,2024-09,False,None,None,purely financial education article,NaN
1,2023-07-14,More education needed on pesticides use to combat environmental destruction,https://dailynews.co.tz/more-education-needed-on-pesticides-use-to-combat-environmental-destruction/,Daily News,2026-05-20,2023-07,False,None,None,purely environmental article,NaN
2,2024-12-03,Bridging gaps: The Samia Infrastructure Bond’s role in national growth,https://dailynews.co.tz/bridging-gaps-the-samia-infrastructure-bonds-role-in-national-growth/,Daily News,2026-05-20,2024-12,True,Investment,Positive,new infrastructure bond for growth,NaN
3,2023-01-13,Kurasini Park project stalls as government shifts focus,https://www.thecitizen.co.tz/tanzania/news/national/kurasini-park-project-stalls-as-government-shifts-focus-4085188,The Citizen,2026-05-20,2023-01,False,None,None,purely politics article,NaN
4,2022-03-03,Revealed: Why air tickets are expensive in East Africa,https://www.thecitizen.co.tz/tanzania/news/national/revealed-why-air-tickets-are-expensive-in-east-africa-3735814,The Citizen,2026-05-20,2022-03,True,General,Neutral,analysis of air ticket prices,NaN
...,...,...,...,...,...,...,...,...,...,...,...
995,2023-08-21,"TBS, LGAs sign deal on food, premises registration",https://dailynews.co.tz/tbs-lgas-sign-deal-on-food-premises-registration/,Daily News,2026-05-20,2023-08,True,General,Neutral,"food, premises registration deal",NaN
996,2024-02-23,"AfDB to finance SGR project linking Tanzania to Burundi, DRC",https://dailynews.co.tz/afdb-to-finance-sgr-project-linking-tanzania-to-burundi-drc/,Daily News,2026-05-20,2024-02,True,Investment,Neutral,SGR project financing by AfDB,NaN
997,2024-04-09,BoT: Rate hike won’t affect banks’ lending,https://dailynews.co.tz/bot-rate-hike-wont-affect-banks-lending/,Daily News,2026-05-20,2024-04,True,Policy,Neutral,"rate hike, banks lending unaffected",NaN
998,2022-11-29,Dar among first seven to trade under AfCFTA rules,https://dailynews.co.tz/dar-among-first-seven-to-trade-under-afcfta-rules/,Daily News,2026-05-20,2022-11,True,Trade,Neutral,AfCFTA rules trading,NaN
